In [13]:
import pandas as pd
import os

df = pd.read_csv("data/train.csv")
print(f"Loaded from CSV: {df.shape}")

df.head()


Loaded from CSV: (3461, 5)


,query_text,user_region_res5,user_region_res4,radius_bucket_3,total_orders
0,amc,8544a103fffffff,8444a11ffffffff,2,500
1,amc,8526e38bfffffff,8426e39ffffffff,2,500
2,amc,8526ee2ffffffff,8426ee3ffffffff,1,500
3,amc,8544e5affffffff,8444e5bffffffff,1,500
4,amc,852a1273fffffff,842a127ffffffff,1,500


In [14]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, mean_absolute_error
from sklearn.base import BaseEstimator, TransformerMixin
from sentence_transformers import SentenceTransformer

class SentenceTransformerWrapper(BaseEstimator, TransformerMixin):
    """Wrapper for sentence-transformers to plug into sklearn pipelines."""
    def __init__(self, model_name="all-MiniLM-L6-v2", device="cpu"):
        self.model_name = model_name
        self.device = device
        self.model = None
    
    def fit(self, X, y=None):
        if self.model is None:
            self.model = SentenceTransformer(self.model_name, device=self.device)
        return self
    
    def transform(self, X):
        if isinstance(X, pd.Series):
            texts = X.tolist()
        else:
            texts = list(X)
        embeddings = self.model.encode(texts, show_progress_bar=False)
        return embeddings


X = df[["query_text", "user_region_res5", "user_region_res4"]]
y = df["radius_bucket_3"].astype(int)

df["sample_weight"] = np.log1p(df["total_orders"])

X_train, X_val, y_train, y_val, w_train, w_val = train_test_split(
    X,
    y,
    df["sample_weight"],
    test_size=0.2,
    random_state=42,
    stratify=y,
)

text_features = "query_text"
cat_features = ["user_region_res5", "user_region_res4"]

preprocess = ColumnTransformer(
    transformers=[
        ("text", SentenceTransformerWrapper(model_name="all-MiniLM-L6-v2"), text_features),
        ("cat", OneHotEncoder(handle_unknown="ignore", sparse_output=False), cat_features),
    ],
    remainder="drop",
)

from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, mean_absolute_error

model = Pipeline(
    steps=[
        ("prep", preprocess),
        ("clf", LogisticRegression(
            max_iter=1000,
            n_jobs=-1,
            class_weight=None
        )),
    ]
)

print("Training (3 classes NEAR/MID/FAR)...")
model.fit(X_train, y_train, clf__sample_weight=w_train)

y_pred = model.predict(X_val)
print("\nValidation (3 classes):")
print(classification_report(y_val, y_pred))

mae_buckets = mean_absolute_error(y_val, y_pred)
print(f"\nMAE on 3-class bucket_id (0..2): {mae_buckets:.3f}")

Training (3 classes NEAR/MID/FAR)...


/Users/zphilipp/miniconda3/lib/python3.12/site-packages/transformers/tokenization_utils_base.py:1601: FutureWarning: `clean_up_tokenization_spaces` was not set. It will be set to `True` by default. This behavior will be depracted in transformers v4.45, and will be then set to `False` by default. For more details check this issue: https://github.com/huggingface/transformers/issues/31884
  warnings.warn(
huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)
huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the envir


Validation (3 classes):
              precision    recall  f1-score   support

           0       0.72      0.73      0.72       192
           1       0.80      0.85      0.83       376
           2       0.75      0.58      0.65       125

    accuracy                           0.77       693
   macro avg       0.76      0.72      0.73       693
weighted avg       0.77      0.77      0.77       693


MAE on 3-class bucket_id (0..2): 0.266


In [15]:
import joblib

MODEL_PATH = "query_region_radius_model_st.joblib"
joblib.dump(model, MODEL_PATH)
print(f"Model saved to {MODEL_PATH}")

Model saved to query_region_radius_model_st.joblib


In [16]:
def predict_radius_segment(query_text, user_region_res5, user_region_res4):
    row = {
        "query_text": str(query_text),
        "user_region_res5": user_region_res5 or "UNKNOWN",
        "user_region_res4": user_region_res4 or "UNKNOWN",
    }
    X_input = pd.DataFrame([row])
    pred_segment = int(model.predict(X_input)[0])
    prob_vec = model.predict_proba(X_input)[0]
    # Confidence is the probability of the predicted class
    confidence = float(prob_vec[pred_segment])
    return {"predicted_segment": pred_segment, "confidence": confidence}


In [ ]:
import pandas as pd

# Test queries - same as in the original notebook
all_rows = [
    # --- test_rows ---
    {"query_text": "massage",          "user_region_res5": "852a1423fffffff", "user_region_res4": "842a143ffffffff"},
    {"query_text": "masage",           "user_region_res5": "852a1423fffffff", "user_region_res4": "842a143ffffffff"},
    {"query_text": "massage",          "user_region_res5": "8544f003fffffff", "user_region_res4": "8444f01ffffffff"},
    {"query_text": "facial",           "user_region_res5": "852b9bc7fffffff", "user_region_res4": "842b9bdffffffff"},
    {"query_text": "smog check",       "user_region_res5": "8528308bfffffff", "user_region_res4": "8428309ffffffff"},
    {"query_text": "oil change",       "user_region_res5": "85269607fffffff", "user_region_res4": "8426961ffffffff"},
    {"query_text": "trampoline park",  "user_region_res5": "852a1073fffffff", "user_region_res4": "842a107ffffffff"},
    {"query_text": "escape room",      "user_region_res5": "8526640ffffffff", "user_region_res4": "8426641ffffffff"},
    {"query_text": "zoo",              "user_region_res5": "8529a54ffffffff", "user_region_res4": "8429a55ffffffff"},
    {"query_text": "sky zone",         "user_region_res5": "8526640ffffffff", "user_region_res4": "8426641ffffffff"},
    {"query_text": "bowling",          "user_region_res5": "85262cd7fffffff", "user_region_res4": "84262cdffffffff"},
    {"query_text": "seaworld",         "user_region_res5": "852986bbfffffff", "user_region_res4": "842986bffffffff"},
    {"query_text": "great wolf lodge", "user_region_res5": "85283471fffffff", "user_region_res4": "8428347ffffffff"},
    {"query_text": "great wolf",       "user_region_res5": "85283471fffffff", "user_region_res4": "8428347ffffffff"},
    {"query_text": "citypass",         "user_region_res5": "8529ab9bfffffff", "user_region_res4": "8429ab9ffffffff"},
    {"query_text": "whale watching",   "user_region_res5": "8529a54ffffffff", "user_region_res4": "8429a55ffffffff"},
    {"query_text": "hotel",            "user_region_res5": "852a1423fffffff", "user_region_res4": "842a143ffffffff"},
    {"query_text": "hotl",             "user_region_res5": "852a1423fffffff", "user_region_res4": "842a143ffffffff"},
    {"query_text": "windows 11",       "user_region_res5": "8529ab9bfffffff", "user_region_res4": "8429ab9ffffffff"},
    {"query_text": "microsoft office", "user_region_res5": "8529ab9bfffffff", "user_region_res4": "8429ab9ffffffff"},
    {"query_text": "costco",           "user_region_res5": "8529ab9bfffffff", "user_region_res4": "8429ab9ffffffff"},
    {"query_text": "cosco",            "user_region_res5": "8529ab9bfffffff", "user_region_res4": "8429ab9ffffffff"},
    {"query_text": "windows 10",       "user_region_res5": "8529ab9bfffffff", "user_region_res4": "8429ab9ffffffff"},
    {"query_text": "airport parking", "user_region_res5": "8529b6d3fffffff", "user_region_res4": "8429b6dffffffff"},
    {"query_text": "architecture boat tour", "user_region_res5": "852664cffffffff", "user_region_res4": "842664dffffffff"},
    {"query_text": "architecture boat tour", "user_region_res5": "852664c3fffffff", "user_region_res4": "842664dffffffff"},
    {"query_text": "bared monkey", "user_region_res5": "852a1073fffffff", "user_region_res4": "842a107ffffffff"},
    {"query_text": "bible museum", "user_region_res5": "852aa847fffffff", "user_region_res4": "842aa85ffffffff"},
    {"query_text": "big air", "user_region_res5": "8544da87fffffff", "user_region_res4": "8444da9ffffffff"},
    {"query_text": "big air trampoline park", "user_region_res5": "8544d84ffffffff", "user_region_res4": "8444d85ffffffff"},
    {"query_text": "laser hair removal", "user_region_res5": "852664c3fffffff", "user_region_res4": "842664dffffffff"},
]

# Test DataFrame and predictions
df_test = pd.DataFrame(all_rows)

# Model expects columns: query_text, user_region_res5, user_region_res4
probs = model.predict_proba(df_test[["query_text", "user_region_res5", "user_region_res4"]])
pred_segments = model.predict(df_test[["query_text", "user_region_res5", "user_region_res4"]])

# Confidence is the probability of the predicted class
df_out = df_test.copy()
df_out["predicted_segment"] = pred_segments.astype(int)
df_out["confidence"] = [float(probs[i][pred_segments[i]]) for i in range(len(pred_segments))]

pd.set_option("display.max_colwidth", None)
pd.set_option("display.max_rows", None)

df_out


,query_text,user_region_res5,user_region_res4,predicted_segment,confidence
0,massage,852a1423fffffff,842a143ffffffff,1,0.753329
1,masage,852a1423fffffff,842a143ffffffff,1,0.826119
2,massage,8544f003fffffff,8444f01ffffffff,1,0.634269
3,facial,852b9bc7fffffff,842b9bdffffffff,0,0.547356
4,smog check,8528308bfffffff,8428309ffffffff,0,0.978104
5,oil change,85269607fffffff,8426961ffffffff,1,0.829892
6,trampoline park,852a1073fffffff,842a107ffffffff,1,0.448563
7,escape room,8526640ffffffff,8426641ffffffff,1,0.914407
8,zoo,8529a54ffffffff,8429a55ffffffff,1,0.966753
9,sky zone,8526640ffffffff,8426641ffffffff,1,0.937813
